In [1]:
import sys
sys.path.append('..')

import os
import re
import glob
import pandas as pd
from nnspike.utils import extract_video_frames
from nnspike.data import create_label_dataframe, sort_by_frames_number, label_dataset_by_opencv, label_dataset_by_model, augment_dataset, set_spike_status
from nnspike.constants import ROI_CNN

## Extract Frames from Videos

In [2]:


def get_all_avi_files(directory_path="C:/Users/MSAD/github/nnspike/storage/20250811/videos/"):
    """
    Get all AVI files from the specified directory with their timestamps.
    
    Args:
        directory_path (str): Path to the directory containing AVI files
    
    Returns:
        list: List of tuples containing (file_path, timestamp)
    """
    # Use glob to find all .avi files in the directory
    avi_files = glob.glob(os.path.join(directory_path, "*.avi"))
    avi_files = [path.replace("\\", "/") for path in avi_files]
    
    # Sort the files for consistent ordering
    avi_files.sort()
    
    # Extract timestamps and create tuples
    result = []
    for avi_file in avi_files:
        # Extract filename without extension
        filename = os.path.basename(avi_file)
        filename_no_ext = os.path.splitext(filename)[0]
        
        # Extract timestamp from filename (assuming format: timestamp_picamera.avi)
        # This will extract the part before '_picamera'
        timestamp_match = re.match(r'^(\d{14})_.*', filename_no_ext)
        if timestamp_match:
            timestamp = timestamp_match.group(1)
        else:
            # If timestamp pattern not found, use the full filename without extension
            timestamp = filename_no_ext
        
        result.append((avi_file, timestamp))
    
    return result

def extract_frames_from_avi_files(avi_files_with_timestamps, base_output_dir="C:/Users/MSAD/github/nnspike/storage/20250811/frames/"):
    """
    Extract frames from all AVI files and save them to folders named by timestamp.
    
    Args:
        avi_files_with_timestamps (list): List of tuples containing (file_path, timestamp)
        base_output_dir (str): Base directory where frame folders will be created
    
    Returns:
        list: List of tuples containing (output_directory, timestamp)
    """
    output_directories_with_timestamps = []
    
    for avi_file, timestamp in avi_files_with_timestamps:
        # Create output directory path
        output_dir = os.path.join(base_output_dir, timestamp)
        
        # Create directory if it doesn't exist
        os.makedirs(output_dir, exist_ok=True)
        
        # Add trailing slash for extract_video_frames function
        output_dir_with_slash = output_dir + "/"
        
        filename = os.path.basename(avi_file)
        print(f"Extracting frames from {filename} to {output_dir_with_slash}")
        
        try:
            # Extract frames using the nnspike utility function
            extract_video_frames(avi_file, output_dir_with_slash)
            print(f"✓ Successfully extracted frames to {timestamp}/")
            # Add successful output directory and timestamp to the list
            output_directories_with_timestamps.append((output_dir, timestamp))
        except Exception as e:
            print(f"✗ Error extracting frames from {filename}: {str(e)}")
    
    return output_directories_with_timestamps

# Get all AVI files with their timestamps
avi_files_with_timestamps = get_all_avi_files(directory_path="C:/Users/MSAD/github/nnspike/storage/20250811/videos/")
print(f"Found {len(avi_files_with_timestamps)} AVI files:")
for file_path, timestamp in avi_files_with_timestamps:
    print(f"  - {os.path.basename(file_path)} -> {timestamp}")

# Extract frames from all AVI files
if avi_files_with_timestamps:
    print("\nStarting frame extraction...")
    output_dirs_with_timestamps = extract_frames_from_avi_files(avi_files_with_timestamps, base_output_dir="C:/Users/MSAD/github/nnspike/storage/20250811/frames/")
    print("\nFrame extraction completed!")
    print(f"Successfully created {len(output_dirs_with_timestamps)} output directories:")
    for output_dir, timestamp in output_dirs_with_timestamps:
        print(f"  - {output_dir} (timestamp: {timestamp})")
else:
    print("No AVI files found to process.")
    output_dirs_with_timestamps = []

avi_files_with_timestamps

Found 1 AVI files:
  - 20250811172540_picamera.avi -> 20250811172540

Starting frame extraction...
Extracting frames from 20250811172540_picamera.avi to C:/Users/MSAD/github/nnspike/storage/20250811/frames/20250811172540/
Frames extracted to: C:\Users\MSAD\github\nnspike\storage\20250811\frames\20250811172540
✓ Successfully extracted frames to 20250811172540/

Frame extraction completed!
Successfully created 1 output directories:
  - C:/Users/MSAD/github/nnspike/storage/20250811/frames/20250811172540 (timestamp: 20250811172540)


[('C:/Users/MSAD/github/nnspike/storage/20250811/videos/20250811172540_picamera.avi',
  '20250811172540')]

In [ ]:
for output_dir, timestamp in output_dirs_with_timestamps:
    print(f"Output directory: {output_dir} (timestamp: {timestamp})")

    label_df = create_label_dataframe(output_dir +"/*", 'right')
    label_df = sort_by_frames_number(label_df)
    label_df = label_dataset_by_opencv(label_df, ROI_CNN, 80)

    status_df = pd.read_csv(f"C:/Users/MSAD/github/nnspike/storage/20250811/sensor_data/{timestamp}_sensor_log.csv")
    df = set_spike_status(label_df, status_df)

    # Export to a csv file
    # df.to_csv(f"../storage/labels/{timestamp}_label.csv", index=False)

## Data Augmentation

In [ ]:
augment_dataset(df, 0.5, "../storage/frames/test_aug1")  # The function returns a dataframe